# Extraction

Sign in as the admin test user, download accepted PDFs from the private bucket, and run `pipeline.extractor.extract` on each one. This notebook does not embed.
Set `SHOW_TEXT = True` to print one chunk. The last cell deletes the temporary files.


In [ ]:
import os
import sys
import tempfile
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
os.chdir(ROOT)
for entry in (ROOT / "notebooks", ROOT):
    text = str(entry)
    if text not in sys.path:
        sys.path.insert(0, text)

load_dotenv(ROOT / ".env")

from helpers import download_supabase_pdfs, preview, timed

SHOW_TEXT = False
workspace = tempfile.TemporaryDirectory(prefix="nb-extract-")

with timed("download") as setup:
    bucket, accepted, rejected = download_supabase_pdfs(Path(workspace.name))

print(bucket)
print(f"download: {setup['seconds']:.3f}s")
print(f"accepted: {len(accepted)}  rejected: {len(rejected)}")
for name, _size, reason in rejected:
    print(f"{name}\trejected: {reason}")

In [ ]:
from pipeline.extractor import extract

documents = []
failures = []
for path in accepted:
    try:
        with timed("extract") as timing:
            document = await extract(path)
        if document.metadata.total_chunks == 0:
            failures.append((path.name, "RuntimeError", "produced no text"))
            continue
        documents.append((document, timing["seconds"]))
    except Exception as exc:
        failures.append((path.name, type(exc).__name__, str(exc)))

In [ ]:
import statistics

print("filename\tpages\tchunks\theadings\tmin\tmedian\tmax\tseconds")
for document, seconds in documents:
    lengths = [len(chunk.text) for chunk in document.chunks]
    headings = {tuple(chunk.headings) for chunk in document.chunks if chunk.headings}
    print(
        document.metadata.filename,
        document.metadata.total_pages,
        document.metadata.total_chunks,
        len(headings),
        min(lengths),
        statistics.median(lengths),
        max(lengths),
        f"{seconds:.3f}",
        sep="\t",
    )

for name, exc_type, message in failures:
    print(f"{name}\tfailure\t{exc_type}\t{message}")

In [ ]:
if SHOW_TEXT and documents:
    chunk = documents[0][0].chunks[0]
    print(chunk.page, chunk.headings)
    print(preview(chunk.text, SHOW_TEXT))

In [ ]:
workspace.cleanup()